In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

In [ ]:
data2 = gpd.read_file(
    "/home/kangah/Desktop/GIS_programming/Geospatial/data/data2.dbf-20250420T151602Z-001/data2.dbf"
)
data2

In [ ]:
Real = data2.iloc[:, 2:13]
Real

In [ ]:
missing_percent = (Real.isnull().sum() / len(Real)) * 100
print(missing_percent)

In [ ]:
Real_clean = Real.dropna()
Real_clean

In [ ]:
# # Real_clean.columns = ['Velocity', 'Top_Wetness_Index', 'Precipitation', 'LULC', 'DistanceFromFault', 'DistanceFromRoad', 'DistanceFromRiver','DEM' 'Geology', 'Aspect']
# Real_clean.rename(
#     columns={
#         "velocity": "Velocity",
#         "TWI": "Top_Wetness_Index",
#         "extract_prec1": "Precipitation",
#         "extract_lulc1": "LULC",
#         "eucdist_faul1": "DistanceFromFault",
#         "distanceFromRoad": "DistanceFromRoad",
#         "distanceFromriver": "DistanceFromRiver",
#         "dem": "DEM",
#         "Geology_CONUS_Clip_PolygonToRaster1": "Geology",
#         "Aspect_DEM2": "Aspect",
#     },
#     inplace=True,
# )
# Real_clean

In [ ]:
pearson_correlation_matrix = Real_clean.corr(method="pearson")

plt.figure(figsize=(12, 10))
sns.heatmap(
    pearson_correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True
)
plt.title("Pearson Correlation Matrix Heatmap")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
X = Real_clean.drop(columns=["velocity"])
y = Real_clean["velocity"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()

## By Kangah (Surveyor, Civil and Geospatial Engineer)

In [ ]:
X_train.count()

In [ ]:
y_train.count()

In [ ]:
X_test.count()

In [ ]:
y_test.count()

In [ ]:
x_train = scaler.fit_transform(X_train)
x_test = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
RS_model = RandomForestRegressor(n_estimators=100, random_state=42)
RS_model.fit(x_train, y_train)
y_pred = RS_model.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")  # By Kangah

In [ ]:
# Get feature importances from the model
importances = RS_model.feature_importances_
feature_names = X.columns

# Create DataFrame
gini_df = pd.DataFrame(
    {"Feature": feature_names, "Importance": importances}
).sort_values(by="Importance", ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    gini_df["Feature"], gini_df["Importance"], color="coral", alpha=0.8, height=0.4
)

# Add central black dot
for i, imp in enumerate(gini_df["Importance"]):
    ax.plot(imp, i, "ko")

# Add a box showing the method
ax.text(
    0.95,
    0.05,
    "■ Mean Decrease Gini",
    transform=ax.transAxes,
    fontsize=12,
    verticalalignment="bottom",
    horizontalalignment="right",
    color="OrangeRed",
)

# Labels
ax.set_xlabel("Mean Decrease in Gini (Feature Importance)", fontsize=12)
ax.set_ylabel("Land Susceptibility Influencing Factors", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Combine the actual and predicted values with coordinates
results_df = pd.DataFrame(
    {
        "Longitude": data2.loc[y_test.index, "long"],
        "Latitude": data2.loc[y_test.index, "lat"],
        "Actual Velocity": y_test.values,
        "Predicted Velocity": y_pred,
    }
)

# Display the DataFrame
print(results_df)

In [ ]:
results_df.to_csv("final_results.csv", index=False)

In [ ]:
# from sklearn.inspection import permutation_importance

# # Evaluate permutation importance
# result = permutation_importance(
#     RS_model, x_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
# )

# # Create DataFrame
# perm_df = pd.DataFrame(
#     {
#         "Feature": X.columns,
#         "Importance": result.importances_mean,
#         "Std": result.importances_std,
#     }
# ).sort_values(by="Importance", ascending=True)

# # Create plot
# fig, ax = plt.subplots(figsize=(10, 6))

# # Plot bars with error bars
# ax.barh(
#     perm_df["Feature"],
#     perm_df["Importance"],
#     xerr=perm_df["Std"],
#     alpha=0.7,
#     height=0.4,
#     color="coral",
# )

# # Add label with square bullet
# ax.text(
#     0.95,
#     0.05,
#     "■ Mean Decrease Accuracy",
#     transform=ax.transAxes,
#     fontsize=12,
#     verticalalignment="bottom",
#     horizontalalignment="right",
#     color="OrangeRed",
# )

# # Labels
# ax.set_xlabel("Mean Decrease in Accuracy (Permutation Importance)")
# ax.set_title("Permutation Feature Importance")

# plt.tight_layout()
# plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# Evaluate permutation importance
result = permutation_importance(
    RS_model, x_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

# Create DataFrame
perm_df = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": result.importances_mean,
        "Std": result.importances_std,
    }
).sort_values(by="Importance", ascending=True)

# Display the DataFrame
print(perm_df)

In [ ]:
from xgboost import XGBRegressor

# Initialize the XGBoost Regressor
xgb_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=100, random_state=42
)

# Fit the model to the training data
xgb_model.fit(x_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(x_test)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)
print(f"XGBoost Mean Squared Error: {mse_xgb}")
print(f"XGBoost R^2 Score: {r2_xgb}")

In [ ]:
y_train